# Categorical MIWAE course benchmark

Each categorical feature has its own observed vocabulary and categorical likelihood. Missing inputs are masked. Imputation takes the most probable category after averaging the predicted class probabilities using importance weights. For mixed datasets, the same latent vector also drives a Student’s t head for each numerical feature.

Choose the dataset near the top, then run the cells in order. To follow a single minibatch, read **Forward pass**, **MIWAE loss**, and **Training loop**. Plotting functions live in the shared figure module.

## Data and settings

Run from the project directory. Mixed dataset entries below identify categorical columns by zero-based position.

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
import json
import math
import time

from miwae_figures import comparison_galleries, diagnostics, version_current, version_figures
from filelock import FileLock
import fastexcel
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

DATA = Path.cwd() / "Course Project Datasets"
OUTPUT = DATA.parent / "outputs" / "miwae"
CATEGORICAL = {"TTTTEG", "HOV", "MUSH", "Splice", "C4"}
MIXED = {
    "CREDIT": [0, 3, 4, 5, 6, 8, 9, 11, 12],
    "Abalone": [0],
    "Adult": [1, 3, 5, 6, 7, 8, 9, 13],
    "Aheart": [4],
    "KDD": [1, 2, 3],
}

In [ ]:
@dataclass(frozen=True)
class Settings:
    seed: int = 1234
    hidden: int = 32
    latent: int = 4
    train_samples: int = 4
    impute_samples: int = 8
    batch: int = 256
    min_updates: int = 64
    min_epochs: int = 1
    learning_rate: float = 0.002


def datasets(group="all"):
    names = sorted(
        p.stem for p in (DATA / "Original Datasets Without Labels").glob("*.xlsx")
    )
    groups = {
        "all": names,
        "numerical": [n for n in names if n not in MIXED and n not in CATEGORICAL],
        "categorical": [n for n in names if n in CATEGORICAL or n in MIXED],
    }
    return groups[group]


def read_frame(path):
    batch = fastexcel.read_excel(path).load_sheet(
        0, header_row=None, skip_rows=0, schema_sample_rows=None, eager=True
    )
    frame = batch.to_pandas()
    frame.columns = [f"x{i+1}" for i in range(frame.shape[1])]
    return frame

In [ ]:
GROUP = "categorical"
print("Available datasets:", ", ".join(datasets(GROUP)))

## Select the run

Use a dataset name and a mask such as C_10 for a single fit. Set either selector to None to run all its available choices. Completed variants with matching settings are reused. Choose a fresh OUTPUT_DIR to retrain.

In [ ]:
DATASET = None  # e.g. "HOV"
VARIANT = None  # e.g. "C_10"
OUTPUT_DIR = OUTPUT
SETTINGS = Settings(min_epochs=1, min_updates=64)

## Prepare the observed inputs

Numerical columns come first in the model input. Standardize them using observed values and keep the original scale for decoding. For categorical columns, keep the observed vocabulary so predicted class indices can be converted back to labels.

In [ ]:
def prepare_inputs(frame, categorical):
    numeric = [j for j in range(frame.shape[1]) if j not in categorical]
    order = numeric + categorical
    mask = frame.iloc[:, order].notna().to_numpy()
    if not mask.any(axis=0).all():
        raise ValueError("A feature has no observations from which to learn")
    values = frame.iloc[:, numeric].to_numpy(dtype=float)
    center, scale = np.nanmean(values, axis=0), np.nanstd(values, axis=0)
    scale[scale == 0] = 1
    encoded = np.zeros(mask.shape, dtype=np.float32)
    n = len(numeric)
    encoded[:, :n] = np.nan_to_num((values - center) / scale)
    labels = []
    for i, j in enumerate(categorical):
        codes, vocabulary = pd.factorize(frame.iloc[:, j])
        encoded[:, n + i] = np.maximum(codes, 0)
        labels.append(vocabulary.to_numpy())
    x = torch.from_numpy(encoded)
    m = torch.from_numpy(mask.astype(np.float32))
    return x, m, numeric, center, scale, labels

## Forward pass

The encoder produces the location and scale of the Gaussian posterior. For each row, sample latent vectors and decode them into Student’s t parameters and category probabilities. The forward pass returns these draws for use in either training or imputation.

In [ ]:
@dataclass
class ModelDraws:
    latent: torch.Tensor              # [importance samples, rows, latent dimensions]
    noise: torch.Tensor               # standard-normal draw used to sample the posterior
    posterior_scale: torch.Tensor
    numeric_means: torch.Tensor
    numeric_dist: torch.distributions.StudentT | None
    category_log_probs: list          # one [samples, rows, classes] tensor per feature


class MIWAE(nn.Module):
    def __init__(self, numeric, levels, cfg):
        super().__init__()
        self.numeric = numeric
        self.levels = levels
        width = numeric + sum(levels)

        def network(inputs, outputs):
            return nn.Sequential(
                nn.Linear(inputs, cfg.hidden),
                nn.ReLU(),
                nn.Linear(cfg.hidden, cfg.hidden),
                nn.ReLU(),
                nn.Linear(cfg.hidden, outputs),
            )

        self.encoder = network(width + numeric + len(levels), 2 * cfg.latent)
        self.decoder = network(cfg.latent, 3 * numeric + sum(levels))

    def forward(self, x, mask, samples):
        numeric = self.numeric
        one_hot_features = []
        for i, size in enumerate(self.levels):
            column = numeric + i
            one_hot = F.one_hot(x[:, column].long(), size).float()
            one_hot_features.append(one_hot * mask[:, column, None])

        inputs = torch.cat([x[:, :numeric], *one_hot_features, mask], dim=-1)
        location, raw_scale = self.encoder(inputs).chunk(2, dim=-1)
        scale = F.softplus(raw_scale) + 1e-4
        noise = torch.randn(samples, *location.shape, device=x.device)
        latent = location + scale * noise
        decoded = self.decoder(latent)

        means = decoded[..., :numeric]
        numeric_dist = None
        if numeric:
            scales = 0.001 + F.softplus(decoded[..., numeric : 2 * numeric])
            degrees = 3 + F.softplus(decoded[..., 2 * numeric : 3 * numeric])
            numeric_dist = torch.distributions.StudentT(degrees, means, scales, validate_args=False)

        category_logits = decoded[..., 3 * numeric :]
        category_log_probs = []
        if self.levels:
            for logits in category_logits.split(self.levels, dim=-1):
                category_log_probs.append(logits.log_softmax(dim=-1))

        return ModelDraws(latent, noise, scale, means, numeric_dist, category_log_probs)


## MIWAE loss

Each latent draw receives a weight from the prior, the encoder posterior, and the likelihood of the observed entries. The weight helper is also used during imputation.

In [ ]:
def log_importance_weights(draws, x, mask):
    """One log weight per latent draw and row, shared by training and imputation."""
    # log p(z) - log q(z | observed x), using the reparameterized Gaussian draw.
    log_weights = (draws.posterior_scale.log()
                   + 0.5 * (draws.noise.square() - draws.latent.square())).sum(dim=-1)

    # Add log p(observed x | z). Missing cells contribute zero.
    numeric = draws.numeric_means.shape[-1]
    if draws.numeric_dist is not None:
        log_density = draws.numeric_dist.log_prob(x[:, :numeric])
        log_weights = log_weights + (log_density * mask[:, :numeric]).sum(dim=-1)

    for i, log_probs in enumerate(draws.category_log_probs):
        column = numeric + i
        target = x[:, column].long()[None, :, None].expand(log_probs.size(0), -1, 1)
        log_probability = log_probs.gather(-1, target).squeeze(-1)
        log_weights = log_weights + log_probability * mask[:, column]
    return log_weights

For each row, take the log of the mean importance weight over the K draws. Minimize the negative average of that bound across rows. The log-sum-exp form keeps this calculation stable.

In [ ]:
def miwae_loss(model, x, mask, samples):
    """Negative importance-weighted bound averaged over the rows in a minibatch."""
    draws = model(x, mask, samples)
    log_weights = log_importance_weights(draws, x, mask)
    return -torch.logsumexp(log_weights, dim=0).mean() + math.log(samples)

## Training loop

Each minibatch calls miwae_loss, backpropagates its gradient, and takes one Adam step. Whole shuffled epochs continue until the configured epoch and update minimums are met.

In [ ]:
def train_model(model, x, mask, cfg, loss_trace):
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    epochs = max(
        cfg.min_epochs, math.ceil(cfg.min_updates / math.ceil(len(x) / cfg.batch))
    )
    total_steps = epochs * math.ceil(len(x) / cfg.batch)
    trace_interval = max(1, math.ceil(total_steps / 200))
    step = 0
    for _ in range(epochs):
        for idx in torch.randperm(len(x)).split(cfg.batch):
            optimizer.zero_grad()
            loss = miwae_loss(model, x[idx], mask[idx], cfg.train_samples)
            step += 1
            if loss_trace is not None and (
                step == 1 or step % trace_interval == 0 or step == total_steps
            ):
                loss_trace.append([step, float(loss.detach())])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 10, error_if_nonfinite=True)
            optimizer.step()
    return epochs

## Impute missing cells

Normalize the importance weights across draws. Use the weighted conditional mean for a numerical feature and the most probable class for a categorical feature.

In [ ]:
@torch.no_grad()
def predict_missing(model, x, mask, cfg):
    """Average numerical predictions and category probabilities over latent draws."""
    predictions = np.empty(x.shape, dtype=np.float32)
    numeric = model.numeric
    for start in range(0, len(x), cfg.batch):
        batch = slice(start, start + cfg.batch)
        draws = model(x[batch], mask[batch], cfg.impute_samples)
        log_weights = log_importance_weights(draws, x[batch], mask[batch])
        weights = log_weights.softmax(dim=0).unsqueeze(-1)
        predictions[batch, :numeric] = (weights * draws.numeric_means).sum(dim=0).numpy()

        for i, log_probs in enumerate(draws.category_log_probs):
            probabilities = (weights * log_probs.exp()).sum(dim=0)
            predictions[batch, numeric + i] = probabilities.argmax(dim=-1).numpy()
    return predictions

In [ ]:
def impute(frame, categorical, cfg, loss_trace=None):
    torch.manual_seed(cfg.seed)
    x, mask, numeric, center, scale, labels = prepare_inputs(frame, categorical)
    model = MIWAE(len(numeric), [len(vocab) for vocab in labels], cfg)
    epochs = train_model(model, x, mask, cfg, loss_trace)
    predictions = predict_missing(model, x, mask, cfg)

    result = frame.copy()
    for i, column in enumerate(numeric):
        missing = frame.iloc[:, column].isna().to_numpy()
        result.iloc[missing, column] = predictions[missing, i] * scale[i] + center[i]
    for i, column in enumerate(categorical):
        missing = frame.iloc[:, column].isna().to_numpy()
        codes = predictions[missing, len(numeric) + i].astype(int)
        result.iloc[missing, column] = labels[i][codes]
    return result, epochs

## Evaluate against the complete workbook

NRMS uses the Frobenius norm of the complete numerical matrix as its denominator. AE is the fraction of hidden categorical values recovered exactly. The reference values enter here for scoring.

In [ ]:
def score(frame, filled, truth, categorical):
    if (
        frame.shape != truth.shape
        or filled.isna().any().any()
        or truth.isna().any().any()
    ):
        raise ValueError("Check workbook alignment and missing values")
    residual = norm = 0.0
    hits = count = 0
    for j in range(frame.shape[1]):
        obs = frame.iloc[:, j].notna().to_numpy()
        actual = truth.iloc[:, j].to_numpy()
        estimate = filled.iloc[:, j].to_numpy()
        if j in categorical:
            aligned = np.array_equal(actual[obs], estimate[obs])
            hits += np.count_nonzero(actual[~obs] == estimate[~obs])
            count += (~obs).sum()
        else:
            aligned = np.allclose(actual[obs], estimate[obs], rtol=1e-12, atol=1e-12)
            residual += np.square(actual - estimate).sum()
            norm += np.square(actual).sum()
        if not aligned:
            raise ValueError(
                f"Observed values disagree with reference in feature {j+1}"
            )
    return dict(
        nrms=float(np.sqrt(residual / norm)) if norm else None,
        ae=float(hits / count) if count else None,
        categorical_missing=int(count),
    )

## Run the selected workbooks

For each supplied mask, fit one model or resume its saved result. Write the completed data and scores before drawing the figures.

In [ ]:
def run(
    dataset=None,
    group="all",
    variant=None,
    cfg=Settings(),
    output=OUTPUT,
    visualizations=True,
):
    changed = set()
    names = datasets(group)
    if dataset:
        alias = {"ttteg": "tttteg"}.get(dataset.lower(), dataset.lower())
        names = [n for n in names if n.lower() == alias]
        if not names:
            raise ValueError(f"Choose a dataset from {datasets(group)}")
    torch.set_num_threads(1)
    for name in names:
        truth = None
        folder = DATA / "Incomplete Datasets Without Labels" / name
        paths = sorted(folder.glob(f"{name}_*.xlsx"))
        if variant:
            selection = [variant] if isinstance(variant, str) else variant
            chosen = {f"{name}_{v}".lower() for v in selection}
            paths = [p for p in paths if p.stem.lower() in chosen]
            if len(paths) != len(chosen):
                raise ValueError(
                    f"Unknown variant for {name}: {chosen - {p.stem.lower() for p in paths}}"
                )
        if not paths:
            raise ValueError(f"No matching input in {folder}")
        dest = Path(output) / "imputed" / name
        dest.mkdir(parents=True, exist_ok=True)
        with FileLock(str(dest / ".train.lock")):
            for path in paths:
                record = dest / (path.stem + ".json")
                parquet = record.with_suffix(".parquet")
                cached = record.exists() and parquet.exists()
                if cached:
                    info = json.loads(record.read_text())
                    if info["settings"] != asdict(cfg):
                        raise ValueError(
                            "Use a different output folder for changed settings"
                        )
                    if not visualizations or version_current(output, record):
                        continue
                start = time.monotonic()
                frame = read_frame(path)
                if truth is None:
                    truth = read_frame(
                        DATA / "Original Datasets Without Labels" / f"{name}.xlsx"
                    )
                    if name == "Splice":
                        # Reference DNA symbols have leading spaces absent in the mask files.
                        truth = truth.apply(lambda col: col.str.strip())
                # Excel may omit fully blank boundary rows/columns from its used range.
                if frame.shape[0] > truth.shape[0] or frame.shape[1] > truth.shape[1]:
                    raise ValueError(f"Input exceeds reference dimensions: {path}")
                frame = frame.reindex(index=truth.index, columns=truth.columns)
                cat = (
                    list(range(frame.shape[1]))
                    if name in CATEGORICAL
                    else MIXED.get(name, [])
                )
                if cached:
                    filled = pd.read_parquet(parquet)
                else:
                    loss_trace = []
                    filled, epochs = impute(frame, cat, cfg, loss_trace=loss_trace)
                    metrics = score(frame, filled, truth, cat)
                    filled.to_parquet(parquet.with_suffix(".part"), index=False)
                    parquet.with_suffix(".part").replace(parquet)
                    info = dict(
                        variant=path.stem,
                        settings=asdict(cfg),
                        epochs=epochs,
                        loss_trace=loss_trace,
                        seconds=round(time.monotonic() - start, 3),
                        **metrics,
                    )
                    record.with_suffix(".tmp").write_text(
                        json.dumps(info, allow_nan=False)
                    )
                    record.with_suffix(".tmp").replace(record)
                    print(path.stem, metrics, flush=True)
                if visualizations:
                    version_figures(
                        output, info, diagnostics(frame, filled, truth, cat)
                    )
                    changed.add(name)
    if visualizations:
        return comparison_galleries(output, changed)

## Fill the grading table

Match saved scores to the rows of the course template and write the workbook under OUTPUT_DIR.

In [ ]:
def report_rows(output=OUTPUT):
    """Collect completed metrics in the order used by the course template."""
    from openpyxl import load_workbook

    book = load_workbook(DATA / "Table-NRMS-AE.xlsx", read_only=True)
    try:
        identifiers = [
            row[0]
            for row in book.active.values
            if isinstance(row[0], str) and len(row[0].rsplit("_", 2)) == 3
        ]
    finally:
        book.close()

    def key(identifier):
        return identifier.casefold().replace("tttteg_", "ttteg_")

    records = {
        key(record.stem): json.loads(record.read_text())
        for record in (Path(output) / "imputed").glob("*/*.json")
        if record.with_suffix(".parquet").exists()
    }
    if records.keys() - {key(identifier) for identifier in identifiers}:
        raise ValueError("Found results without a corresponding template row")
    rows = {}
    for identifier in identifiers:
        info = records.get(key(identifier))
        if info is None:
            continue
        cfg = info["settings"]
        details = (
            f"h={cfg['hidden']}, d={cfg['latent']}, K={cfg['train_samples']}, "
            f"L={cfg['impute_samples']}, batch={cfg['batch']}, epochs={info['epochs']}, "
            f"Adam lr={cfg['learning_rate']}, seed={cfg['seed']}"
        )
        ae = info["ae"]
        if ae is None and info["variant"].rsplit("_", 2)[0] in MIXED:
            ae = "N/A"
            details += ", 0 hidden categorical cells"
        rows[identifier] = [info["nrms"], ae, details]
    return rows


COMPLEXITY = (
    "For a fixed network, training O(E × N × K) and imputation O(N × L). "
    "Batch activations scale with batch size and sample count. Input/output storage scales with N × features."
)

In [ ]:
def export_table(output=OUTPUT):
    """Refresh the Excel report from every finished run in this output folder."""
    from openpyxl import load_workbook
    from openpyxl.styles import Alignment

    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    target = output / "Table-NRMS-AE.xlsx"
    with FileLock(str(output / ".report.lock")):
        rows = report_rows(output)
        book = load_workbook(DATA / target.name)
        sheet = book.active
        for cell in sheet["A"]:
            name = cell.value
            if name in rows:
                for col, value in enumerate(rows[name], 2):
                    sheet.cell(cell.row, col, value)
            if name == "Average NRMS/AE":
                for col in ["B", "C"]:
                    sheet[f"{col}{cell.row}"] = (
                        f'=IFERROR(AVERAGE({col}2:{col}{cell.row-1}),"")'
                    )
            if name == "Complexity Analysis":
                sheet.cell(cell.row, 4, COMPLEXITY).alignment = Alignment(
                    wrap_text=True
                )
                sheet.row_dimensions[cell.row].height = 45
        temp = target.with_suffix(".tmp.xlsx")
        book.save(temp)
        temp.replace(target)
    print(f"{len(rows)} completed variant(s) included in {target}")
    return target

## Execute the selected experiment

In [ ]:
gallery = run(dataset=DATASET, group=GROUP, variant=VARIANT, cfg=SETTINGS, output=OUTPUT_DIR, visualizations=True)
workbook = export_table(OUTPUT_DIR)
from IPython.display import FileLink, display
import os
display(FileLink(os.path.relpath(gallery), result_html_prefix="Browse report figures: "))

## Inspect an imputation

The headerless source columns appear in their original order as x1, x2, and so on. NRMS divides the numerical reconstruction error by the norm of the complete numerical matrix. AE is exact-match accuracy over hidden categorical cells. The workbook combines both notebooks’ completed runs.

In [ ]:
import pandas as pd
from pathlib import Path

example = DATASET or "HOV"
folder = next(p for p in (Path(OUTPUT_DIR) / "imputed").iterdir()
              if p.name.casefold().replace("tttteg", "ttteg") == example.casefold().replace("tttteg", "ttteg"))
selection = VARIANT or "C_10"
pd.read_parquet(folder / f"{folder.name}_{selection.upper()}.parquet").head()